# Análise — Mario DRL (Fase 1)

Carrega os CSVs de todos os treinos (DQN, PPO, A2C × 4 fases × 3 seeds = 36)
e produz **tudo** que vai entrar na Seção 3 (Experimentos e Resultados) do artigo:

- Métricas Grupo I (recompensa final, AUC, sample efficiency, estabilidade)
- Métricas Grupo II (taxa de conclusão, distância, mortes, tempo)
- Análise estatística (Spearman + Mann-Whitney)
- Gráficos: curvas de aprendizado, comparação por fase, dificuldade canônica
- Visualizações: 3 algoritmos lado-a-lado, evolução temporal

## 1. Instalação

In [ ]:
# ===================================================================
# Instalação — Colab (Python 3.10 ou 3.11) ou Linux local
# ===================================================================
# IMPORTANTE: este notebook requer Python 3.10 ou 3.11.
#   - nes-py 8.2.1 quebra em Python 3.12 (OverflowError em uint8)
#   - No Colab, garanta que está em Python 3.11 antes de rodar.
#
# Fluxo:
#   1) Rode esta célula UMA VEZ.
#   2) RESTART o kernel/sessão.
#   3) Rode a partir da célula 2.
# ===================================================================

import sys

PY = sys.version_info
print(f"Python: {PY.major}.{PY.minor}.{PY.micro}")
print(f"Executable: {sys.executable}")

if PY >= (3, 12):
    raise RuntimeError(
        f"\n⚠ Python {PY.major}.{PY.minor} não é suportado.\n"
        "   nes-py 8.2.1 quebra em Python 3.12+ (OverflowError em uint8).\n"
        "   No Colab: Runtime → Change runtime type → escolha Python 3.10 ou 3.11.\n"
        "   Localmente: use venv com Python 3.10/3.11."
    )

print(f"\n✓ Python {PY.major}.{PY.minor} é compatível. Instalando dependências (~3 min)...\n")

# Instalações progressivas — se algo falhar, fica claro qual passo foi
print(">> [1/5] numpy<2.0 (nes-py 8.2.1 quebra com NumPy 2.x)")
!pip install --quiet "numpy<2.0"

print(">> [2/5] RL stack (gym, gymnasium, shimmy, SB3)")
!pip install --quiet "gym==0.26.2" "gymnasium==0.29.1" "shimmy==1.3.0" "stable-baselines3==2.3.2"

print(">> [3/5] Super Mario Bros env (nes-py compila Cython, ~60s)")
!pip install --quiet "gym-super-mario-bros==7.4.0" "nes-py==8.2.1"

print(">> [4/5] Visualização (imageio, opencv)")
!pip install --quiet "imageio>=2.30" "opencv-python-headless>=4.8"

print(">> [5/5] PyTorch + utilitários (no-op se já tiver no Colab)")
!pip install --quiet "torch>=2.0,<2.5" "pandas" "scipy" "matplotlib" "seaborn" "tqdm" "tensorboard"

print("\n" + "="*60)
print("✓ Instalação concluída.")
print()
print("⚠  AGORA RESTART A SESSÃO antes de continuar:")
print("    Colab: Runtime → Restart session  (Ctrl+M .)")
print("    Local: Restart Kernel")
print("="*60)

## 2. Imports

In [ ]:
import os, json, time, random, pickle, warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# RL stack
import gymnasium as gym
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from nes_py.wrappers import JoypadSpace

# Stable-Baselines3
from stable_baselines3 import DQN, PPO, A2C
from stable_baselines3.common.atari_wrappers import MaxAndSkipEnv, WarpFrame
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import (
    DummyVecEnv, SubprocVecEnv, VecFrameStack, VecMonitor
)
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, CallbackList

# Stats
from scipy.stats import spearmanr, mannwhitneyu

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
sns.set_theme(style="whitegrid", context="paper")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python : {os.sys.version_info.major}.{os.sys.version_info.minor}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Storage compartilhado com os notebooks de treino

In [ ]:
# ----------------------------------------------------------------------
# Storage: auto-detecta Colab → monta Drive; senão usa pasta local.
# Os 4 notebooks compartilham o MESMO diretório de resultados.
# ----------------------------------------------------------------------
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)
    ROOT_DIR = Path("/content/drive/MyDrive/mario_drl_results")
    print(f"Colab detectado — usando Drive: {ROOT_DIR}")
except ImportError:
    IN_COLAB = False
    ROOT_DIR = Path("./mario_drl_results").resolve()
    print(f"Local — usando: {ROOT_DIR}")

MODELS_DIR  = ROOT_DIR / "models"
LOGS_DIR    = ROOT_DIR / "logs"
TB_DIR      = ROOT_DIR / "tensorboard"
METRICS_DIR = ROOT_DIR / "metrics"
PLOTS_DIR   = ROOT_DIR / "plots"
for d in (MODELS_DIR, LOGS_DIR, TB_DIR, METRICS_DIR, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 4. Configuração global

In [ ]:
# ----------------------------------------------------------------------
# Parâmetros globais do experimento
# ----------------------------------------------------------------------
SMOKE_TEST = True   # True = teste rápido (~2 min); False = experimento real

STAGES = ["1-1", "1-2", "4-1", "8-1"]
SEEDS  = [42, 123, 2024]
ALGOS  = ["DQN", "PPO", "A2C"]

if SMOKE_TEST:
    TOTAL_TIMESTEPS = 10_000
    EVAL_FREQ       = 2_000
    N_EVAL_EPISODES = 2
    STAGES_TO_RUN   = ["1-1"]
    SEEDS_TO_RUN    = [42]
else:
    TOTAL_TIMESTEPS = 500_000
    EVAL_FREQ       = 10_000
    N_EVAL_EPISODES = 5
    STAGES_TO_RUN   = STAGES
    SEEDS_TO_RUN    = SEEDS

FRAME_STACK = 4
print(f"Modo: {'SMOKE_TEST' if SMOKE_TEST else 'COMPLETO'}  |  "
      f"Treinamentos: {len(ALGOS) * len(STAGES_TO_RUN) * len(SEEDS_TO_RUN)}  |  "
      f"Timesteps/treino: {TOTAL_TIMESTEPS:,}")

## 5. Hiperparâmetros (referência — usado nas visualizações)

In [ ]:
HPARAMS = {
    "DQN": dict(
        learning_rate     = 1e-4,
        buffer_size       = 100_000,
        learning_starts   = 10_000,
        batch_size        = 32,
        tau               = 1.0,
        gamma             = 0.99,
        train_freq        = 4,
        gradient_steps    = 1,
        target_update_interval = 10_000,
        exploration_fraction   = 0.10,
        exploration_initial_eps = 1.0,
        exploration_final_eps   = 0.01,
        max_grad_norm     = 10.0,
        n_envs            = 1,
    ),
    "PPO": dict(
        learning_rate     = 2.5e-4,
        n_steps           = 128,
        batch_size        = 256,
        n_epochs          = 4,
        gamma             = 0.99,
        gae_lambda        = 0.95,
        clip_range        = 0.1,
        ent_coef          = 0.01,
        vf_coef           = 0.5,
        max_grad_norm     = 0.5,
        n_envs            = 8,
    ),
    "A2C": dict(
        learning_rate     = 7e-4,
        n_steps           = 5,
        gamma             = 0.99,
        gae_lambda        = 1.0,
        ent_coef          = 0.01,
        vf_coef           = 0.25,
        max_grad_norm     = 0.5,
        rms_prop_eps      = 1e-5,
        use_rms_prop      = True,
        n_envs            = 16,
    ),
}

## 6. Env factory (necessário p/ visualizações da seção 14)

In [ ]:
# Force shim V21 — JoypadSpace usa API antiga (gym pre-0.26)
import gym as _legacy_gym
from gym.wrappers import TimeLimit as _GymTimeLimit
from shimmy import GymV21CompatibilityV0 as _GymCompat
print(f"gym {_legacy_gym.__version__} → shimmy.{_GymCompat.__name__} (forçado V21)")


def _strip_time_limit(env):
    """
    Remove o TimeLimit que gym.make() envelopa automaticamente em 0.26+.
    Esse wrapper espera API nova (5-tuple step), mas SuperMarioBrosEnv
    retorna 4-tuple. Removendo-o, deixamos o env "cru" passar pro JoypadSpace.
    """
    while isinstance(env, _GymTimeLimit):
        env = env.env
    return env


class CompatJoypadSpace(JoypadSpace):
    """JoypadSpace tolerante a (seed, options) no reset."""
    def reset(self, seed=None, options=None, **kwargs):
        return super().reset(**kwargs)


def make_mario_env(stage: str = "1-1", seed: int = 0):
    env_id = f"SuperMarioBros-{stage}-v0"
    env = gym_super_mario_bros.make(env_id)
    env = _strip_time_limit(env)            # remove TimeLimit antes do JoypadSpace
    env = CompatJoypadSpace(env, SIMPLE_MOVEMENT)
    env = _GymCompat(env=env)               # converte API antiga → gymnasium nova
    env = MaxAndSkipEnv(env, skip=4)
    env = WarpFrame(env, width=84, height=84)
    env = Monitor(env)
    env.action_space.seed(seed)
    return env


def make_vec_env_mario(stage: str, n_envs: int, seed: int, use_subproc: bool = True):
    def make_one(rank):
        def _init():
            return make_mario_env(stage=stage, seed=seed + rank)
        return _init
    env_fns = [make_one(i) for i in range(n_envs)]
    if n_envs > 1 and use_subproc:
        vec_env = SubprocVecEnv(env_fns, start_method="fork")
    else:
        vec_env = DummyVecEnv(env_fns)
    vec_env = VecFrameStack(vec_env, n_stack=FRAME_STACK)
    vec_env = VecMonitor(vec_env)
    return vec_env


def set_global_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## 7. Carregamento dos logs

In [ ]:
def load_all_logs() -> pd.DataFrame:
    """Concatena todos os CSVs em LOGS_DIR num DataFrame longo."""
    rows = []
    for csv_path in sorted(LOGS_DIR.glob("*.csv")):
        name = csv_path.stem
        try:
            algo, stage_part, seed_part = name.split("_")
            stage = stage_part.replace("stage", "")
            seed = int(seed_part.replace("seed", ""))
        except ValueError:
            continue
        df = pd.read_csv(csv_path)
        df["algo"] = algo; df["stage"] = stage; df["seed"] = seed
        rows.append(df)
    if not rows:
        print("Nenhum log encontrado em LOGS_DIR.")
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)


df_all = load_all_logs()
print(f"Total de avaliações: {len(df_all):,}")
if len(df_all) > 0:
    print(f"Configurações distintas: {df_all.groupby(['algo','stage','seed']).ngroups}")
    print(f"Algoritmos: {sorted(df_all['algo'].unique())}")
    print(f"Fases    : {sorted(df_all['stage'].unique())}")
    print(f"Seeds    : {sorted(df_all['seed'].unique())}")
    display(df_all.head())

## 8. Grupo I — comparação entre arquiteturas

In [ ]:
def compute_group1_metrics(df_all, final_window=50_000):
    if df_all.empty: return pd.DataFrame()
    rows = []
    for (algo, stage, seed), g in df_all.groupby(["algo", "stage", "seed"]):
        g = g.sort_values("timestep")
        per_t = g.groupby("timestep")["reward"].mean().reset_index().sort_values("timestep")
        max_t = per_t["timestep"].max(); max_r = per_t["reward"].max()
        final_mask = per_t["timestep"] >= (max_t - final_window)
        R_final = per_t.loc[final_mask, "reward"].mean()
        sigma_final = per_t.loc[final_mask, "reward"].std()
        auc = np.trapz(per_t["reward"], per_t["timestep"]) / max(max_t, 1)
        above = per_t[per_t["reward"] >= 0.8 * max_r]
        t80 = int(above["timestep"].iloc[0]) if len(above) > 0 else np.nan
        rows.append(dict(algo=algo, stage=stage, seed=seed,
                         R_final=R_final, AUC=auc, t80=t80, sigma_final=sigma_final))
    return pd.DataFrame(rows)


def aggregate_group1(metrics_g1):
    return (metrics_g1.groupby(["algo", "stage"])
        .agg(R_final_median=("R_final", "median"),
             R_final_iqr=("R_final", lambda x: x.quantile(0.75) - x.quantile(0.25)),
             AUC_median=("AUC", "median"),
             t80_median=("t80", "median"),
             sigma_final_median=("sigma_final", "median"))
        .reset_index())


if len(df_all) > 0:
    metrics_g1 = compute_group1_metrics(df_all)
    metrics_g1.to_csv(METRICS_DIR / "group1_per_seed.csv", index=False)
    metrics_g1_agg = aggregate_group1(metrics_g1)
    metrics_g1_agg.to_csv(METRICS_DIR / "group1_aggregated.csv", index=False)
    print("=== Grupo I — agregado por (algo, stage) ===\n")
    display(metrics_g1_agg.round(2))

## 9. Grupo II — avaliação automática de dificuldade

In [ ]:
STAGE_LENGTH = {"1-1": 3266, "1-2": 3266, "4-1": 3866, "8-1": 3266}


def compute_group2_metrics(df_all):
    if df_all.empty: return pd.DataFrame()
    rows = []
    for (algo, stage, seed), g in df_all.groupby(["algo", "stage", "seed"]):
        last_t = g["timestep"].max()
        last_eval = g[g["timestep"] == last_t]
        tau = last_eval["flag_get"].mean()
        d_bar = (last_eval["max_x_pos"] / STAGE_LENGTH.get(stage, 3266)).mean()
        deaths_mean = last_eval["deaths"].mean()
        successful = last_eval[last_eval["flag_get"] == 1]
        time_mean = successful["frames"].mean() if len(successful) > 0 else np.nan
        rows.append(dict(algo=algo, stage=stage, seed=seed,
                         tau=tau, d_bar=d_bar, deaths=deaths_mean, time_frames=time_mean))
    return pd.DataFrame(rows)


def aggregate_group2(metrics_g2):
    return (metrics_g2.groupby(["algo", "stage"])
        .agg(tau_median=("tau", "median"),
             d_bar_median=("d_bar", "median"),
             deaths_median=("deaths", "median"),
             time_median=("time_frames", "median"))
        .reset_index())


if len(df_all) > 0:
    metrics_g2 = compute_group2_metrics(df_all)
    metrics_g2.to_csv(METRICS_DIR / "group2_per_seed.csv", index=False)
    metrics_g2_agg = aggregate_group2(metrics_g2)
    metrics_g2_agg.to_csv(METRICS_DIR / "group2_aggregated.csv", index=False)
    print("=== Grupo II — agregado por (algo, stage) ===\n")
    display(metrics_g2_agg.round(3))

## 10. Spearman: ranking do agente vs dificuldade canônica

In [ ]:
def spearman_difficulty(metrics_g2_agg):
    if metrics_g2_agg.empty: return pd.DataFrame()
    rank_canonical = {s: i for i, s in enumerate(STAGES, start=1)}
    rows = []
    for algo in metrics_g2_agg["algo"].unique():
        sub = metrics_g2_agg[metrics_g2_agg["algo"] == algo].copy()
        sub["rank_canonical"] = sub["stage"].map(rank_canonical)
        sub = sub.sort_values("rank_canonical")
        if len(sub) < 3: continue
        for metric, sign in [("tau_median","-"),("d_bar_median","-"),
                             ("deaths_median","+"),("time_median","+")]:
            x = sub["rank_canonical"].values
            y = sub[metric].values
            if np.all(np.isnan(y)): continue
            rho, p = spearmanr(x, y, nan_policy="omit")
            rows.append(dict(algo=algo, metric=metric, expected_sign=sign,
                             rho=rho, p_value=p, n=len(sub)))
    return pd.DataFrame(rows)


if len(df_all) > 0:
    spearman_df = spearman_difficulty(metrics_g2_agg)
    spearman_df.to_csv(METRICS_DIR / "spearman_difficulty.csv", index=False)
    print("=== Spearman: ranking do agente vs dificuldade canônica ===\n")
    display(spearman_df.round(3))

## 11. Mann-Whitney U entre algoritmos

In [ ]:
from itertools import combinations

def mannwhitney_between_algos(metrics_g1, metric="R_final"):
    if metrics_g1.empty: return pd.DataFrame()
    rows = []
    for stage in metrics_g1["stage"].unique():
        sub = metrics_g1[metrics_g1["stage"] == stage]
        for a, b in combinations(sorted(sub["algo"].unique()), 2):
            xa = sub[sub["algo"] == a][metric].dropna().values
            xb = sub[sub["algo"] == b][metric].dropna().values
            if len(xa) < 2 or len(xb) < 2: continue
            stat, p = mannwhitneyu(xa, xb, alternative="two-sided")
            rows.append(dict(stage=stage, comparison=f"{a} vs {b}",
                             median_diff=float(np.median(xa) - np.median(xb)),
                             U=stat, p_value=p, significant=bool(p < 0.05)))
    return pd.DataFrame(rows)


if len(df_all) > 0:
    mw_df = mannwhitney_between_algos(metrics_g1, metric="R_final")
    mw_df.to_csv(METRICS_DIR / "mannwhitney_algos.csv", index=False)
    print("=== Mann-Whitney U (R_final, por fase) ===\n")
    display(mw_df.round(4))

## 12. Plot — curvas de aprendizado

In [ ]:
def plot_learning_curves(df_all, savedir=PLOTS_DIR):
    if df_all.empty: print("Sem dados."); return
    stages_present = sorted(df_all["stage"].unique())
    n = len(stages_present)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4), sharey=False)
    if n == 1: axes = [axes]
    palette = {"DQN":"#1f77b4", "PPO":"#ff7f0e", "A2C":"#2ca02c"}
    for ax, stage in zip(axes, stages_present):
        sub = df_all[df_all["stage"] == stage]
        for algo, g in sub.groupby("algo"):
            agg = (g.groupby("timestep")["reward"]
                   .agg(["median", lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
                   .rename(columns={"<lambda_0>":"q25","<lambda_1>":"q75"}).reset_index())
            color = palette.get(algo)
            ax.plot(agg["timestep"], agg["median"], label=algo, color=color, linewidth=2)
            ax.fill_between(agg["timestep"], agg["q25"], agg["q75"], alpha=0.2, color=color)
        ax.set_title(f"Fase {stage}")
        ax.set_xlabel("Timesteps"); ax.set_ylabel("Recompensa (mediana ± IQR)")
        ax.legend(loc="best", fontsize=9); ax.grid(alpha=0.3)
    plt.tight_layout()
    out = savedir / "learning_curves.png"
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
    print(f"✓ {out}")


if len(df_all) > 0:
    plot_learning_curves(df_all)

## 13. Plot — comparação Grupo I

In [ ]:
def plot_group1_comparison(metrics_g1, savedir=PLOTS_DIR):
    if metrics_g1.empty: print("Sem dados."); return
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    palette = {"DQN":"#1f77b4", "PPO":"#ff7f0e", "A2C":"#2ca02c"}
    for ax, metric, title in zip(axes, ["R_final","AUC","t80"],
        ["Recompensa final média","AUC normalizada","Timesteps até 80% do máximo"]):
        sns.barplot(data=metrics_g1, x="stage", y=metric, hue="algo",
                    palette=palette, ax=ax, errorbar=("ci", 95),
                    order=sorted(metrics_g1["stage"].unique()))
        ax.set_title(title); ax.set_xlabel("Fase"); ax.set_ylabel(metric)
        ax.legend(title="Algoritmo")
    plt.tight_layout()
    out = savedir / "group1_comparison.png"
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
    print(f"✓ {out}")


if len(df_all) > 0:
    plot_group1_comparison(metrics_g1)

## 14. Plot — Grupo II por fase

In [ ]:
def plot_group2_difficulty(metrics_g2, savedir=PLOTS_DIR):
    if metrics_g2.empty: print("Sem dados."); return
    titles = {"tau":"Taxa de conclusão (τ)",
              "d_bar":"Distância normalizada (d̄)",
              "deaths":"Mortes por episódio"}
    palette = {"DQN":"#1f77b4", "PPO":"#ff7f0e", "A2C":"#2ca02c"}
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    stage_order = [s for s in STAGES if s in metrics_g2["stage"].unique()]
    for ax, (metric, title) in zip(axes, titles.items()):
        for algo, g in metrics_g2.groupby("algo"):
            agg = g.groupby("stage")[metric].agg(["median",
                lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
            agg.columns = ["median","q25","q75"]
            agg = agg.reindex(stage_order)
            x = np.arange(len(stage_order))
            ax.plot(x, agg["median"], "o-", label=algo, linewidth=2,
                    markersize=8, color=palette.get(algo))
            ax.fill_between(x, agg["q25"], agg["q75"], alpha=0.2, color=palette.get(algo))
        ax.set_xticks(x); ax.set_xticklabels(stage_order)
        ax.set_xlabel("Fase (ordem canônica →)"); ax.set_ylabel(title); ax.set_title(title)
        ax.legend(title="Algoritmo"); ax.grid(alpha=0.3)
    plt.tight_layout()
    out = savedir / "group2_difficulty.png"
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
    print(f"✓ {out}")


if len(df_all) > 0:
    plot_group2_difficulty(metrics_g2)

## 15. Comparação visual lado-a-lado

Função genérica `render_models_side_by_side(dict_models, ...)` — recebe
qualquer mapeamento `{label: modelo}` e gera GIF + animação inline com
todos jogando em paralelo (mesma fase, mesmo seed).

In [ ]:
import cv2
import imageio.v2 as imageio
from matplotlib import animation as _mpl_animation
from IPython.display import HTML, display


def animate_frames_inline(frames, fps=15, figsize=(8, 4)):
    if not frames: return HTML("<i>Nenhum frame.</i>")
    fig, ax = plt.subplots(figsize=figsize); ax.axis("off")
    im = ax.imshow(frames[0])
    def update(i):
        im.set_array(frames[i])
        return [im]
    ani = _mpl_animation.FuncAnimation(fig, update, frames=len(frames),
                                       interval=1000.0/fps, blit=True)
    html = ani.to_jshtml(default_mode="loop"); plt.close(fig)
    return HTML(html)


def run_episode_for_render(model, stage="1-1", max_steps=3000, seed=999):
    env_id = f"SuperMarioBros-{stage}-v0"
    base = gym_super_mario_bros.make(env_id, render_mode="rgb_array")
    base = CompatJoypadSpace(base, SIMPLE_MOVEMENT)
    base = _GymCompat(env=base)
    base = MaxAndSkipEnv(base, skip=4)
    base = WarpFrame(base, width=84, height=84)
    base = Monitor(base)
    vec_env = DummyVecEnv([lambda: base])
    vec_env = VecFrameStack(vec_env, n_stack=FRAME_STACK)
    render_env = vec_env.venv.envs[0]
    frames, metrics = [], dict(reward=0.0, max_x=0, deaths=0, flag_get=False, steps=0)
    prev_life = None
    obs = vec_env.reset()
    for _ in range(max_steps):
        f = render_env.render()
        if f is not None: frames.append(f.copy())
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = vec_env.step(action)
        info0 = info[0]
        metrics["reward"] += float(reward[0]); metrics["steps"] += 1
        x = int(info0.get("x_pos", 0))
        if x > metrics["max_x"]: metrics["max_x"] = x
        life = info0.get("life", None)
        if prev_life is not None and life is not None and life < prev_life:
            metrics["deaths"] += 1
        prev_life = life
        if info0.get("flag_get", False): metrics["flag_get"] = True
        if done[0]:
            f = render_env.render()
            if f is not None: frames.append(f.copy())
            break
    vec_env.close()
    return frames, metrics


def _add_label_bar(frame, label_top, metrics_text=""):
    out = frame.copy(); h, w = out.shape[:2]
    cv2.rectangle(out, (0, 0), (w, 26), (0, 0, 0), -1)
    cv2.putText(out, label_top, (8, 19),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
    if metrics_text:
        cv2.rectangle(out, (0, h-22), (w, h), (0, 0, 0), -1)
        cv2.putText(out, metrics_text, (8, h-6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return out


def render_models_side_by_side(models, stage="1-1", max_steps=3000, fps=15,
                                seed=999, out_path=None, show_running_metrics=True):
    labels = list(models.keys())
    print(f"Renderizando {labels} | stage {stage} | seed {seed}")
    all_frames, all_metrics = {}, {}
    for label, model in models.items():
        print(f"  ► {label}...")
        frames, metrics = run_episode_for_render(model, stage, max_steps, seed)
        all_frames[label] = frames; all_metrics[label] = metrics
        flag = "✓" if metrics["flag_get"] else "✗"
        print(f"    {len(frames):4d} frames | r={metrics['reward']:+7.1f} "
              f"| max_x={metrics['max_x']:5d} | flag={flag}")
    max_len = max(len(f) for f in all_frames.values())
    for label in labels:
        f = all_frames[label]
        if len(f) < max_len:
            pad = f[-1] if f else np.zeros((240, 256, 3), dtype=np.uint8)
            f.extend([pad.copy() for _ in range(max_len - len(f))])
    composed = []
    for t in range(max_len):
        panels = []
        for label in labels:
            text = ""
            if show_running_metrics:
                final = all_metrics[label]
                progress = min((t + 1) / max(final["steps"], 1), 1.0)
                text = f"x={int(final['max_x']*progress)} r={final['reward']*progress:+.0f}"
            panels.append(_add_label_bar(all_frames[label][t], label, text))
        composed.append(np.concatenate(panels, axis=1))
    out_path = Path(out_path) if out_path else (PLOTS_DIR / f"comparison_{stage}.gif")
    imageio.mimsave(out_path, composed, duration=int(round(1000.0/fps)), loop=0)
    print(f"\n✓ Salvo: {out_path}")
    return out_path, all_metrics, composed

### 15.1 — DQN vs PPO vs A2C na mesma fase/seed

In [ ]:
# Comparação DQN vs PPO vs A2C — mesma fase, mesmo seed
stage_for_render = "1-1"
seed_for_render = 42

algo_models = {}
for algo, cls in [("DQN", DQN), ("PPO", PPO), ("A2C", A2C)]:
    path = MODELS_DIR / f"{algo}_stage{stage_for_render}_seed{seed_for_render}.zip"
    if path.exists():
        algo_models[algo] = cls.load(path, device=device)
        print(f"✓ {algo}: {path.name}")
    else:
        print(f"✗ {algo} não encontrado: {path.name}")

if len(algo_models) >= 2:
    _, _, composed = render_models_side_by_side(
        algo_models, stage=stage_for_render, max_steps=3000, seed=999,
        out_path=PLOTS_DIR / f"comparison_algos_{stage_for_render}.gif")
    display(animate_frames_inline(composed, fps=15, figsize=(12, 4)))
else:
    print("\n⚠ Treine pelo menos 2 algoritmos antes desta comparação.")

### 15.2 — Evolução temporal de um algoritmo

Mesmo algoritmo em diferentes checkpoints do treino. Visualiza a curva
de aprendizado qualitativamente: 100k → Mario travado; 500k → completa.

In [ ]:
import re
algo_for_evolution = "PPO"
stage_for_evolution = "1-1"
seed_for_evolution = 42

ckpt_dir = MODELS_DIR / "checkpoints"
prefix = f"{algo_for_evolution}_stage{stage_for_evolution}_seed{seed_for_evolution}"
pattern = re.compile(rf"^{re.escape(prefix)}_(\d+)_steps\.zip$")

ckpts = []
if ckpt_dir.exists():
    for p in sorted(ckpt_dir.glob(f"{prefix}_*.zip")):
        m = pattern.match(p.name)
        if m: ckpts.append((int(m.group(1)), p))
ckpts.sort()

if not ckpts:
    print(f"⚠ Nenhum checkpoint para {prefix}. Verifique se rodou treino com SMOKE_TEST=False.")
else:
    n_show = min(5, len(ckpts))
    chosen = ckpts[::max(len(ckpts)//n_show, 1)][:n_show]
    cls = {"DQN": DQN, "PPO": PPO, "A2C": A2C}[algo_for_evolution]
    models = {f"{s//1000}k": cls.load(p, device=device) for s, p in chosen}
    for lbl in models: print(f"  ✓ {lbl}")
    _, _, composed = render_models_side_by_side(
        models, stage=stage_for_evolution, max_steps=2500, seed=999,
        out_path=PLOTS_DIR / f"evolution_{algo_for_evolution}_{stage_for_evolution}.gif")
    display(animate_frames_inline(composed, fps=15, figsize=(12, 4)))

## 16. Artefatos gerados

Em `mario_drl_results/`:

```
metrics/
├── group1_per_seed.csv         # Grupo I por (algo, stage, seed)
├── group1_aggregated.csv       # Grupo I por (algo, stage), mediana ± IQR
├── group2_per_seed.csv
├── group2_aggregated.csv
├── spearman_difficulty.csv     # ρ por algoritmo, por métrica
└── mannwhitney_algos.csv       # U por par-de-algoritmos × fase

plots/
├── learning_curves.png
├── group1_comparison.png
├── group2_difficulty.png
├── comparison_algos_1-1.gif    # DQN vs PPO vs A2C
└── evolution_PPO_1-1.gif       # PPO 100k→500k
```

Esses arquivos vão direto para a Seção 3 do artigo .tex.